# 03 SPEC-DRIVEN\n\n> 核心：AI 生成的代码必须严格遵守显式规范，而非\"我觉得应该这样\"。

## 3.1 问题定义\n\n### AI 生成代码的失控问题\n\n典型场景：\n- AI 生成代码时\"自行决定\"了颜色、字号、参数默认值\n- 这些决定不在需求范围内，却无法追溯来源\n- Code Review 只能发现\"风格问题\"，无法发现\"规范问题\"\n\n**关键洞察**：AI 生成代码时会有大量\"隐性决策\"，这些决策分散在代码各处，没有被显式记录。

In [ ]:
def spec_driven_demo():\n    spec = \"\"\"\n输入约束:\n- email: 必须符合 RFC 5322\n- password: 至少 8 字符，包含数字和字母\n- username: 3-20 字符，字母数字下划线\n\n输出格式:\n- 成功: User(id: int, email: str, username: str)\n- 失败: ErrorResponse(code: str, message: str)\n\n边界条件:\n- 邮箱已注册 → code='EMAIL_EXISTS'\n- 密码不符合要求 → code='INVALID_PASSWORD'\n- 用户名已占用 → code='USERNAME_TAKEN'\n\"\"\"\n    print(\"Spec-Driven 开发：规范先行，AI 在规范约束下生成代码\")\n    print(\"规范定义合约，AI 在合约内工作\")\n    return spec\n\nresult = spec_driven_demo()\nprint(\"\\n生成的规范:\", result)\nprint(\"\\n执行结果: 函数正常运行，打印了规范内容\")

## 3.2 SPEC-DRIVEN 的核心价值\n\n### 三个核心问题\n\n| 问题 | 传统开发 | SPEC-DRIVEN |\n|------|----------|-------------|\n| AI 隐性决策 | 分散在代码各处 | 集中在规范文档 |\n| 规范追溯 | 只能通过代码审查 | 规范即文档 |\n| 边界条件 | 隐含在 if-else | 显式枚举 |

In [ ]:
# 模拟 Spec-Driven 的 AI 调用（实际会调用 AI API）\ndef ai_generate_with_spec(spec):\n    messages = [\n        {\"role\": \"user\", \"content\": (\n            \"根据以下规范，生成 Python 函数 register_user(email, password, username) 的实现代码。\\n\\n规范:\\n\"\n            + spec + \"\\n\\n要求:\\n\"\n            \"1. 参数验证在函数内部完成\\n\"\n            \"2. 使用 dataclass 定义 User 和 ErrorResponse\\n\"\n            \"3. 添加类型注解\\n\"\n            \"4. 返回值使用 Union[User, ErrorResponse]\\n\"\n            \"5. 只返回代码，不要解释\\n\"\n        )}\n    ]\n    print(\"Spec-Driven 生成的代码:\")\n    print(\"=\" * 50)\n    print(\"(实际调用 AI API，AI 在规范约束下生成代码)\")\n\nai_generate_with_spec(\"INPUT_SPEC\")

## 3.3 规范即合约\n\n### 为什么规范是合约？\n\n```\n规范（Spec） = 机器可读 + 人类可读 + 语义明确\n```\n\n**三个属性**：\n1. **机器可读**：AI 可以在 Prompt 中直接使用\n2. **人类可读**：人类可以审查和验证\n3. **语义明确**：每个字段的含义、取值范围、边界条件都有定义\n\n### 合约的力量\n\n- 合约一旦确定，AI 生成代码时就不能\"自行决定\"约束外的值\n- 合约违反 = 明显的错误，而非风格问题\n- 合约可以版本化管理，可以 Diff，可以 Review

In [ ]:
# 验证规范是否被遵守（伪代码）\ndef verify_spec_compliance(code, spec):\n    \"\"\"\n    检查 AI 生成的代码是否遵守了规范合约\n    - dataclass User 和 ErrorResponse 是否定义\n    - 返回类型是否为 Union[User, ErrorResponse]\n    - 边界条件是否覆盖\n    \"\"\"\n    issues = []\n    if \"dataclass\" not in code:\n        issues.append(\"缺少 dataclass 定义\")\n    if \"Union[User, ErrorResponse]\" not in code:\n        issues.append(\"返回类型不符合规范\")\n    if \"EMAIL_EXISTS\" not in code:\n        issues.append(\"缺少 EMAIL_EXISTS 边界条件\")\n    return issues\n\nprint(\"验证规范合规性: issues =\", verify_spec_compliance(\"dummy\", {}))

## 3.4 DESIGN.md：设计规范的机器可读格式\n\n### 为什么需要 DESIGN.md？\n\n设计规范需要同时满足：\n- **人类可读**：设计师、产品经理可以阅读和编写\n- **机器可读**：AI Agent 可以解析并严格遵守\n\n**Markdown + YAML Frontmatter** 兼顾两者：\n\n```yaml\n---\ncolors:\n  primary: '#FF6B6B'\n  secondary: '#4ECDC4'\n  tertiary: '#95E1D3'\n---\n\n# Design System\n\n## Overview\n简洁、现代、注重可读性。\n\n## Colors\n- Primary: {colors.primary} — 品牌色\n```\n\n**核心机制**：令牌引用（{colors.primary}）而非硬编码值，确保 AI 引用规范而非自作主张。

In [ ]:
design_md_content = \"\"\"\n---\ncolors:\n  primary: '#FF6B6B'\n  secondary: '#4ECDC4'\n  tertiary: '#95E1D3'\nfont:\n  heading: 'Inter, sans-serif'\n  body: 'Inter, sans-serif'\n---\n\n# Design System\n\n## Overview\n简洁、现代、注重可读性。颜色活泼但不过于鲜艳。\n\n## Colors\n- Primary: {colors.primary} — 用于按钮、链接等主要交互元素\n- Secondary: {colors.secondary} — 用于次要元素、图标\n- Tertiary: {colors.tertiary} — 用于背景、卡片\n\n## Typography\n- Headings: {font.heading}, 700 weight\n- Body: {font.body}, 400 weight\n\n## Components\n### Button\n- Primary: bg={colors.primary}, text=white\n- Secondary: bg={colors.secondary}, text=black\n\"\"\"\n\n# 模拟 AI 解析 DESIGN.md 并生成代码\nimport re\ncolors = {}\nfor match in re.finditer(r'colors\\.(\\w+):\\s*[\\'\\\"]?([^\\\'\\\"\\n]+)[\\\'\\\"]?', design_md_content):\n    colors[match.group(1)] = match.group(2)\n\nprint(\"从 DESIGN.md 解析的颜色令牌:\", colors)\nprint(\"AI 生成代码时会引用 {colors.primary} 而非 #FF6B6B\")

## 3.5 实验结论\n\n### DESIGN.md 格式验证\n- YAML frontmatter 包含机器可读的设计令牌（颜色、字体、组件属性）\n- Markdown body 包含人类可读的设计说明\n- 令牌引用机制（{colors.tertiary}）确保 AI 生成代码引用规范而非硬编码值\n- 强制章节顺序（Overview → Colors → Typography → Components）\n\n### Spec-Driven 开发流程\n1. 规范先行：明确输入约束、输出格式、边界条件\n2. AI 生成：将规范作为 Prompt 上下文，让 AI 生成符合规范的代码\n3. 人类验证：检查生成代码是否严格遵守规范（而非只看功能是否实现）\n\n### 核心价值\n- 减少 AI\"幻视代码\"：规范是 AI 必须遵守的合约\n- 人类保持控制权：规范定义\"做什么\"，AI 决定\"怎么做\"\n- 可追溯可验证：规范 → 生成 → 验证，每步都有据可查